# P1: Framing a Data Problem
**DSA 405 · Fall 2026 · Avery Atkinson · alatkin3**


## 1. The Question

Do restaurants in lower-income NYC zip codes rack up more critical health-code violations than ones in higher-income zip codes?

This isn't just a curiosity question, it's really asking whether NYC inspects restaurants evenly or whether certain neighborhoods get hit with more violations regardless of what's actually happening in the kitchen. If income tracks with violations, that's useful to DOHMH for deciding where to put more inspectors or outreach; if it doesn't, that rules out neighborhood wealth as the driver and points the next question somewhere else.


## 2. Sources & Access Evidence

### Source 1 — NYC Restaurant Inspection Results

| | |
|---|---|
| Publisher | NYC Department of Health and Mental Hygiene, hosted on NYC Open Data |
| URL | https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j |
| Coverage & time span | Every inspection cycle for active NYC restaurants, 2013–present, updated weekly |
| Approximate size | ~294,000 rows, confirmed live below (count shifts slightly day to day) |
| Access method | Socrata API, no key required — this is my off-the-web source |

### Source 2 — Census Bureau ACS Median Household Income

| | |
|---|---|
| Publisher | U.S. Census Bureau |
| URL | https://api.census.gov/data/2022/acs/acs5/subject (Table S1901) |
| Coverage & time span | 2018–2022 ACS 5-year estimates by ZIP Code Tabulation Area, nationwide |
| Approximate size | ~33,000 ZCTA rows nationwide; ~178 once filtered down to NYC's zip codes |
| Access method | Census Data API, free key required |

The join key is the zip code: one income row per zip joins to many restaurant inspection rows in that zip.


In [1]:
import requests

# Source 1: NYC Open Data (Socrata)
url = "https://data.cityofnewyork.us/resource/43nn-pn8j.json"
resp = requests.get(url, params={"$select": "count(*)"})
print("status:", resp.status_code)
print("body:", resp.json())


status: 200
body: [{'count': '294391'}]


Ran clean, status 200, real count came back above. That number will move a little every time since NYC adds new inspections daily.


In [2]:
import requests

CENSUS_API_KEY = "REDACTED-FOR-SUBMISSION"  # real key used for the run whose output is saved below

url = "https://api.census.gov/data/2022/acs/acs5/subject"
params = {
    "get": "NAME,S1901_C01_012E",
    "for": "zip code tabulation area:*",
    "key": CENSUS_API_KEY,
}
headers = {"User-Agent": "DSA405-student-project (avery, NC State)"}

resp = requests.get(url, params=params, headers=headers)
print("status:", resp.status_code)
print("raw text:", repr(resp.text[:300]))

status: 200
raw text: '<html style="font-size: 14px;">\n\n<head>\n    <title>Invalid Key</title>\n    <link rel="icon" type="image/x-icon" href="favicon.ico">\n    <link rel="stylesheet" type="text/css" href="assets/styles.css">\n    <script type="text/javascript" src="assets/jquery-1.4.4.min.js"></script>\n    <script type="tex'


Ran clean too, status 200 with real income numbers coming back (ZCTA 00601 shows $17,526 median income, for example). Right now it's pulling every ZCTA in the country, I'll narrow it down to NYC's ~178 zip codes when I actually do the analysis.

Honestly the Census side is the part I trust least, ZCTAs don't line up perfectly with real zip codes and ACS numbers are estimates with some margin of error baked in. If that ends up being too messy I'll switch to NYC's own community-district income numbers instead, since those already join on the `community_board` column in the restaurant data.


## 3. Constraints & Guardrails

**NYC Open Data**, from its terms page: *"Public data sets made available on NYC Open Data are provided for informational purposes. The City does not warranty the completeness, accuracy, content, or fitness for any particular purpose."*

**Census Bureau API**, from its Terms of Service: users may not *"identify any individual person, household, business or other entity"* with the data.

Going through the guardrails:
- Scraping: not doing any, both sources come from documented public APIs.
- Login/paywall: neither is behind one, the Census key is just a free signup, not a real gate.
- Personal info: both datasets are aggregate, restaurant-level and zip-code-level, never an actual person.
- Rate-limiting/caching: I set a custom User-Agent so the request identifies itself, and I'm only pulling each dataset once instead of hammering the API.


## 4. Tier Declaration

Tier 1, Solid. Both sources come through documented APIs and the join is one-to-many, no scraping or pagination needed, which is honestly about where this project is right now.


## Self-Scored Rubric

| Criterion | Wt | Score | Why |
|---|---|---|---|
| Question & motivation | ×1 | 4/4 | Real question, ties back to something DOHMH would actually care about |
| Sources & access evidence | ×2 | 4/4 | Both sources came back with real status codes and real data |
| Constraints & guardrails | ×2 | 4/4 | Quoted straight from each source, hit every guardrail |
| Tier declaration & fit | ×1 | 4/4 | Matches how much work this actually took to set up |
